In [3]:
import numpy as np
from collections import defaultdict

class HMMTagger:
    def __init__(self):
        # Define 10 POS tags from the Penn Treebank tagset
        self.states = [
            'NN',   # Noun, singular or mass
            'NNS',  # Noun, plural
            'VB',   # Verb, base form
            'VBD',  # Verb, past tense
            'JJ',   # Adjective
            'RB',   # Adverb
            'IN',   # Preposition/subordinating conjunction
            'DT',   # Determiner
            'CC',   # Coordinating conjunction
            'PRP'   # Personal pronoun
        ]
        self.observations = []  # Words in vocabulary
        self.start_prob = {}  # Initial state probabilities
        self.trans_prob = {}  # Transition probabilities
        self.emit_prob = {}  # Emission probabilities
        self.tag_dict = defaultdict(list)  # Dictionary mapping words to possible tags

    def train(self, tagged_sentences):
        """Train the HMM using tagged sentences."""
        # Extract unique words
        words = set()
        tag_counts = defaultdict(int)
        transition_counts = defaultdict(int)
        emission_counts = defaultdict(int)
        
        # First pass: collect statistics
        for sentence in tagged_sentences:
            prev_tag = None
            for word, tag in sentence:
                # Skip tags not in our set of 10
                if tag not in self.states:
                    continue
                    
                words.add(word)
                tag_counts[tag] += 1
                self.tag_dict[word].append(tag)
                
                if prev_tag:
                    transition_counts[(prev_tag, tag)] += 1
                
                emission_counts[(tag, word)] += 1
                prev_tag = tag
        
        self.observations = list(words)
        
        # Calculate initial probabilities
        total_sentences = len(tagged_sentences)
        initial_counts = defaultdict(int)
        
        for sentence in tagged_sentences:
            if sentence and sentence[0][1] in self.states:
                initial_counts[sentence[0][1]] += 1
        
        for tag in self.states:
            self.start_prob[tag] = (initial_counts[tag] + 1) / (total_sentences + len(self.states))
        
        # Calculate transition probabilities
        for tag1 in self.states:
            for tag2 in self.states:
                count = transition_counts.get((tag1, tag2), 0) + 1  # Add-one smoothing
                self.trans_prob[(tag1, tag2)] = count / (tag_counts[tag1] + len(self.states))
        
        # Calculate emission probabilities
        for tag in self.states:
            for word in self.observations:
                count = emission_counts.get((tag, word), 0) + 1  # Add-one smoothing
                self.emit_prob[(tag, word)] = count / (tag_counts[tag] + len(self.observations))
    
    def viterbi(self, sentence):
        """Use Viterbi algorithm to find most likely tag sequence."""
        V = [{}]  # Viterbi matrix
        path = {}
        
        # Initialize base cases (t == 0)
        for tag in self.states:
            if sentence[0] in self.observations:
                V[0][tag] = self.start_prob[tag] * self.emit_prob[(tag, sentence[0])]
            else:
                # Handle unknown words
                V[0][tag] = self.start_prob[tag] * 0.001
            path[tag] = [tag]
        
        # Run Viterbi for t > 0
        for t in range(1, len(sentence)):
            V.append({})
            newpath = {}
            
            for curr_tag in self.states:
                max_prob = -1
                max_tag = None
                
                for prev_tag in self.states:
                    if sentence[t] in self.observations:
                        prob = V[t-1][prev_tag] * self.trans_prob[(prev_tag, curr_tag)] * self.emit_prob[(curr_tag, sentence[t])]
                    else:
                        # Handle unknown words
                        prob = V[t-1][prev_tag] * self.trans_prob[(prev_tag, curr_tag)] * 0.001
                    
                    if prob > max_prob:
                        max_prob = prob
                        max_tag = prev_tag
                
                V[t][curr_tag] = max_prob
                newpath[curr_tag] = path[max_tag] + [curr_tag]
            
            path = newpath
        
        # Find the best path
        max_prob = -1
        max_tag = None
        
        for tag in self.states:
            if V[len(sentence)-1][tag] > max_prob:
                max_prob = V[len(sentence)-1][tag]
                max_tag = tag
        
        return path[max_tag], max_prob

    def tag(self, sentence):
        """Tag a sentence."""
        words = sentence.split()
        tags, _ = self.viterbi(words)
        return list(zip(words, tags))

# Example usage
if __name__ == "__main__":
    # Sample training data with Penn Treebank tags
    training_data = [
        [("The", "DT"), ("cat", "NN"), ("sat", "VBD"), ("on", "IN"), ("the", "DT"), ("mat", "NN"), (".", ".")],
        [("A", "DT"), ("dog", "NN"), ("barks", "VBZ"), ("loudly", "RB"), (".", ".")],
        [("I", "PRP"), ("saw", "VBD"), ("a", "DT"), ("cat", "NN"), ("yesterday", "NN"), (".", ".")]
    ]
    
    # Initialize and train the tagger
    tagger = HMMTagger()
    tagger.train(training_data)
    
    # Test the tagger
    test_sentence = "The dog sat on the mat"
    tagged_sentence = tagger.tag(test_sentence)
    print(tagged_sentence)
    print('///////////////////')

    print(f'initial state distribution: {tagger.start_prob}')
    print(f'Transition probabilities: {tagger.trans_prob}')
    print(f"Emission probabilities: {tagger.emit_prob}")

[('The', 'DT'), ('dog', 'NN'), ('sat', 'VBD'), ('on', 'IN'), ('the', 'DT'), ('mat', 'NN')]
///////////////////
initial state distribution: {'NN': 0.07692307692307693, 'NNS': 0.07692307692307693, 'VB': 0.07692307692307693, 'VBD': 0.07692307692307693, 'JJ': 0.07692307692307693, 'RB': 0.07692307692307693, 'IN': 0.07692307692307693, 'DT': 0.23076923076923078, 'CC': 0.07692307692307693, 'PRP': 0.15384615384615385}
Transition probabilities: {('NN', 'NN'): 0.13333333333333333, ('NN', 'NNS'): 0.06666666666666667, ('NN', 'VB'): 0.06666666666666667, ('NN', 'VBD'): 0.13333333333333333, ('NN', 'JJ'): 0.06666666666666667, ('NN', 'RB'): 0.13333333333333333, ('NN', 'IN'): 0.06666666666666667, ('NN', 'DT'): 0.06666666666666667, ('NN', 'CC'): 0.06666666666666667, ('NN', 'PRP'): 0.06666666666666667, ('NNS', 'NN'): 0.1, ('NNS', 'NNS'): 0.1, ('NNS', 'VB'): 0.1, ('NNS', 'VBD'): 0.1, ('NNS', 'JJ'): 0.1, ('NNS', 'RB'): 0.1, ('NNS', 'IN'): 0.1, ('NNS', 'DT'): 0.1, ('NNS', 'CC'): 0.1, ('NNS', 'PRP'): 0.1, ('VB